In [6]:
# Configuración del entorno
!pip install wikipedia-api==0.5.4
!pip install sentence-transformers chromadb langchain transformers torch pandas

In [7]:
# 1️⃣ Creación de conjuntos de datos (Wikipedia → CSV)
import wikipediaapi
import pandas as pd
import os

# Crear carpeta data
os.makedirs("data", exist_ok=True)

# Inicializar Wikipedia (versión antigua, sin user_agent)
wiki = wikipediaapi.Wikipedia('en')

# Cargar la página
page = wiki.page("Federated_learning")
if not page.exists():
    raise Exception("La página no existe en Wikipedia.")

# Extraer texto completo
full_text = page.text

# Dividir en segmentos de aproximadamente 300 palabras
words = full_text.split()
chunk_size = 300
chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

# Crear DataFrame
df = pd.DataFrame({
    "id": [f"chunk_{i}" for i in range(len(chunks))],
    "title": ["Federated_learning"] * len(chunks),
    "text": chunks
})

# Guardar CSV
df.to_csv("data/wiki_corpus.csv", index=False)
print("CSV creado con", len(chunks), "chunks")

CSV creado con 15 chunks


In [9]:
# 2️⃣ Incrustaciones + Almacén de vectores (ChromaDB)
from sentence_transformers import SentenceTransformer
import chromadb

# Cargar dataset
df = pd.read_csv("data/wiki_corpus.csv")

# Crear cliente ChromaDB
client = chromadb.Client()
collection = client.create_collection("wiki_ai", get_or_create=True)

# Modelo de embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

# Insertar chunks en ChromaDB
for idx, row in df.iterrows():
    emb = model.encode(row["text"]).tolist()
    collection.add(
        ids=[row["id"]],
        metadatas=[{"title": row["title"]}],
        documents=[row["text"]],
        embeddings=[emb]
    )

print("Chunks insertados en ChromaDB:", len(df))

Chunks insertados en ChromaDB: 15


In [14]:
# 3️⃣ Canalización de consultas (LangChain + RAG)
from langchain.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain.vectorstores import Chroma
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# ---------------------------
# 3.1 — LLM ligero para Colab
# ---------------------------
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512
)

llm = HuggingFacePipeline(pipeline=pipe)

# ---------------------------
# 3.2 — Vectorstore usando ChromaDB
# ---------------------------
vectorstore = Chroma(
    client=client,
    collection_name="wiki_ai",
    embedding_function=None
)

# ---------------------------
# 3.3 — Configurar RetrievalQA
# ---------------------------
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)

# ---------------------------
# 3.4 — Ejemplo de consulta
# ---------------------------
respuesta = qa.invoke({"query": "Explain federated learning challenges in healthcare."})
print("Ejemplo de consulta:", respuesta["result"])

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cuda:0
/tmp/ipython-input-1566462442.py:21: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)
Token indices sequence length is longer than the specified maximum sequence length for this model (1691 > 512). Running this sequence through the model will result in indexing errors


Ejemplo de consulta: a systematic review of Federated Learning in the Healthcare Area: From the Perspective of Data Properties and Applications


In [15]:
# 4️⃣ Generar y guardar resumen (rag_summary.md)
os.makedirs("outputs", exist_ok=True)

# Recuperar los mejores chunks
retrieved_docs = vectorstore.similarity_search("Federated Learning", k=5)
context = "\n\n".join([d.page_content for d in retrieved_docs])

# Prompt para el resumen
summary_prompt = f"""
Generate a coherent 400–500 word summary about Federated Learning based only on the context below:

Context:
{context}
"""

# Generar resumen con el LLM
summary = qa.invoke({"query": summary_prompt})["result"]

# Guardar en Markdown
with open("outputs/rag_summary.md", "w", encoding="utf-8") as f:
    f.write(summary)

print("Resumen guardado en outputs/rag_summary.md")

Resumen guardado en outputs/rag_summary.md


In [16]:
# 5️⃣ Ejemplos de consultas guardadas (retrieval_examples.json)
import json

examples = {
    "query_1": "Explain federated learning challenges in healthcare.",
    "answer_1": qa.invoke({"query": "Explain federated learning challenges in healthcare."})["result"],
    "query_2": "What is the purpose of federated learning?",
    "answer_2": qa.invoke({"query": "What is the purpose of federated learning?"})["result"],
    "query_3": "How does federated learning protect privacy?",
    "answer_3": qa.invoke({"query": "How does federated learning protect privacy?"})["result"]
}

with open("outputs/retrieval_examples.json", "w", encoding="utf-8") as f:
    json.dump(examples, f, indent=4)

print("Ejemplos guardados en outputs/retrieval_examples.json")

Ejemplos guardados en outputs/retrieval_examples.json


In [17]:
# TAREA ADICIONAL: Reflexión conceptual entre el modelo multiagente y enfoque RAG
reflection = """
# Reflexión adicional: Comparación conceptual

## 1️⃣ Manejo de ambigüedad y contradicciones
El flujo de trabajo multiagente maneja la ambigüedad y las contradicciones mediante la colaboración entre varios agentes especializados. Cada agente puede proponer soluciones, validar información y cuestionar inconsistencias. Esto permite debatir diferentes interpretaciones y llegar a una respuesta refinada, especialmente en problemas abiertos o mal definidos.

El enfoque RAG depende de la calidad de los documentos recuperados. Si estos contienen información contradictoria, el modelo puede reflejarla sin resolverla automáticamente. RAG es más literal y factual, pero no razona sobre ambigüedad de manera activa.

## 2️⃣ Veracidad y cobertura de recuperación de datos
RAG enfatiza la veracidad porque las respuestas se basan en documentos recuperados. La cobertura depende del corpus indexado: si algo no está en la base de datos, no será incluido. Los sistemas multiagente pueden generar respuestas más amplias, pero con mayor riesgo de errores o información no verificada.

## 3️⃣ Adecuación según tipo de pregunta
- Preguntas abiertas o exploratorias: flujo multiagente es más adecuado, permite síntesis, debate y razonamiento complejo.
- Preguntas basadas en hechos: RAG es preferible, reduce errores y asegura respuestas basadas en fuentes verificadas.

**Conclusión:** Multiagente → creatividad y razonamiento; RAG → precisión factual y cobertura de información.
"""

with open("outputs/reflection.md", "w", encoding="utf-8") as f:
    f.write(reflection)

print("Reflexión guardada en outputs/reflection.md")

Reflexión guardada en outputs/reflection.md
